# Conditional Diffusion: Segmentation → Brain MRI (seg2MRI)

Segmentation-conditioned 3D DDPM/DDIM trained with classifier-free guidance.
The model receives a 4-class BraTS segmentation mask (BG/NCR/edema/enhancing
tumor, normalized to [-1, 1]) as a channel-concat input alongside the noised
volume, and learns to generate anatomically faithful T1N brain MRI conditioned
on that mask.

## Dependencies (run these first)

Both preprocessing notebooks must finish before this one can train:

- **`data_processing/processing.ipynb`** — produces `Processed/brats/32/`
  (T1N volumes normalized to [-1, 1] at 32³)
- **`data_processing/processing_seg.ipynb`** — produces `Processed/brats/32_seg/`
  (matching segmentation masks with discrete labels {0, 1, 2, 3} at 32³)

The conditional model loads (T1N, seg) pairs by matching filenames between
these two folders.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
from typing import Tuple

import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

UNCOND_SRC = Path.cwd().parent / "unconditional" / "src"
COND_SRC   = Path.cwd() / "src"
sys.path.insert(0, str(UNCOND_SRC))
sys.path.insert(0, str(COND_SRC))

from model.unet3d import UNet3D
from model.ema import EMA
from model.training import (
    get_gpu_memory_nvidia_smi,
    compute_weight_norm, snapshot_weights,
    compute_weight_update_ratio, compute_ema_divergence,
    train_one_epoch, validate,
)
from diffusion.schedule import NoiseScheduler
from sampler.sampler import make_scheduler, sample
from sampler.timestep_sampler import LossAwareTimestepSampler
from utility.stats_tracker import StatsTracker
from conditional_datasets import BraTSCondDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  device={DEVICE}")

In [ ]:
@dataclass
class Config:
    data_path:     str = r"C:\Users\Sven\Desktop\Data\Processed\brats\32"
    seg_data_path: str = r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg"
    output_path:   str = "output_conditional"
    # model
    image_size:            int = 32
    in_channels:           int = 1
    cond_channels:         int = 1
    model_channels:        int = 96
    channel_mult:          Tuple[int, ...] = (1, 2, 4, 4)
    num_res_blocks:        int = 2
    attention_resolutions: Tuple[int, ...] = (8, 4)
    num_groups:            int = 8
    dropout:               float = 0.0
    # diffusion
    num_timesteps:    int = 250
    beta_schedule:    str = "cosine"
    prediction_type:  str = "v"
    # training
    num_epochs:     int = 300
    batch_size:     int = 10
    learning_rate:  float = 2e-4
    warmup_steps:   int = 500
    ema_decay:      float = 0.9999
    grad_clip:      float = 0.5            # tighter than unconditional (1.0) — gradients are larger with cond
    # enhancements
    use_loss_aware_sampling: bool = True
    loss_history_size:       int  = 10
    snr_gamma:               float = 5.0
    # CFG
    cfg_dropout_prob: float = 0.15
    guidance_scale:   float = 3.0
    num_seg_classes:  int   = 4            # BG, NCR, edema, enhancing tumor
    # device
    device: str = DEVICE

    def __post_init__(self):
        os.makedirs(self.output_path, exist_ok=True)
        os.makedirs(f"{self.output_path}/samples", exist_ok=True)
        os.makedirs(f"{self.output_path}/checkpoints", exist_ok=True)


In [ ]:
EXPERIMENT_NAME = "conditional"

EXPERIMENT_META = {
    "model_name": "conditional",
    "experiment_description": "Segmentation-conditioned diffusion model with Classifier-Free Guidance (CFG)",
    "research_question": "Can segmentation-conditioned generation with CFG produce anatomically faithful 3D medical images?",
    "changed_parameters": ["seg_data_path", "cond_channels", "cfg_dropout_prob", "guidance_scale", "grad_clip"],
    "changed_from": ["None", "0", "0.0", "1.0", "1.0"],
    "changed_to": ["32_seg", "1", "0.15", "3.0", "0.5"],
}

SAMPLE_EPOCHS = [1, 10, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300]

config = Config()

print(f"experiment: {EXPERIMENT_NAME}")
print(f"  epochs={config.num_epochs}  batch={config.batch_size}  lr={config.learning_rate}")
print(f"  cond_channels={config.cond_channels}  cfg_dropout={config.cfg_dropout_prob}  guidance={config.guidance_scale}")
print(f"  seg classes: 0=BG, 1=NCR, 2=ED, 3=ET")


In [ ]:
dataset = BraTSCondDataset(config.data_path, config.seg_data_path, config.num_seg_classes)
config.image_size = dataset.volume_shape[0]

val_size = int(len(dataset) * 0.1)
train_size = len(dataset) - val_size
train_subset, val_subset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_subset, batch_size=config.batch_size, shuffle=True,
                          pin_memory=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_subset, batch_size=config.batch_size, shuffle=False,
                        pin_memory=True)

print(f"train: {train_size}  val: {val_size}  batches: {len(train_loader)}")


In [ ]:
model = UNet3D(config).to(config.device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model: {n_params:,} params ({n_params / 1e6:.1f}M)")

with torch.no_grad():
    x = torch.randn(1, 1, config.image_size, config.image_size, config.image_size, device=config.device)
    s = torch.randn(1, 1, config.image_size, config.image_size, config.image_size, device=config.device)
    t = torch.randint(0, config.num_timesteps, (1,), device=config.device)
    assert model(x, t, cond=s).shape == x.shape       # conditional forward
    assert model(x, t, cond=None).shape == x.shape    # null conditional (for CFG)
print("forward pass ok (with and without cond)")


In [ ]:
noise_scheduler = NoiseScheduler(config.num_timesteps, config.beta_schedule, device=config.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
ema = EMA(model, decay=config.ema_decay)
timestep_sampler = LossAwareTimestepSampler(config.num_timesteps, config.loss_history_size)

stats = StatsTracker(EXPERIMENT_META, config)
stats.record_environment()
stats.record_model(model, config)
stats.record_dataset(dataset, train_size, val_size)

preview_scheduler = make_scheduler("ddim", config.beta_schedule, config.prediction_type,
                                   config.num_timesteps)

total_steps = config.num_epochs * len(train_loader)
print(f"optimizer: AdamW (lr={config.learning_rate}, wd=0.01)  total_steps={total_steps}")


In [ ]:
config_dict = {k: (list(v) if isinstance(v, tuple) else v)
               for k, v in config.__dict__.items() if not k.startswith("_")}
with open(f"{config.output_path}/config.json", "w") as f:
    json.dump(config_dict, f, indent=2)
with open(f"{config.output_path}/environment.json", "w") as f:
    json.dump(stats.stats["environment"], f, indent=2)

# fix one (vol, seg) pair for the mid-epoch previews so we can watch the
# same conditioning improve across epochs
_, preview_seg = val_subset[0]
preview_seg = preview_seg.unsqueeze(0).to(config.device)   # (1, 1, D, H, W)

print(f"config + environment saved to {config.output_path}")
print(f"preview cond fixed to val[0] for reproducible epoch comparisons")


In [ ]:
def save_nifti(volume, path):
    if volume.dim() > 3:
        volume = volume.squeeze()
    nib.save(nib.Nifti1Image(volume.cpu().numpy().astype(np.float32), np.eye(4)), path)


def show_sample(volume, title="sample", save_path=None):
    if volume.dim() > 3:
        volume = volume.squeeze()
    v = volume.cpu().numpy()
    d, h, w = v.shape
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(v[d // 2], cmap="gray", vmin=-1, vmax=1)
    axes[0].set_title("axial")
    axes[1].imshow(v[:, h // 2], cmap="gray", vmin=-1, vmax=1)
    axes[1].set_title("coronal")
    axes[2].imshow(v[:, :, w // 2], cmap="gray", vmin=-1, vmax=1)
    axes[2].set_title("sagittal")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    plt.close()


@torch.no_grad()
def preview_with_ema(seed=42):
    ema.apply_shadow(model)
    vol = sample(model, preview_scheduler, config.image_size,
                 num_steps=50, device=config.device, seed=seed,
                 cond=preview_seg, guidance_scale=config.guidance_scale)
    ema.restore(model)
    return vol


In [ ]:
train_losses, val_losses = [], []
global_step = 0
best_val_loss = float("inf")

stats.stats["training_info"]["start_time"] = datetime.now().isoformat()

print(f"training {EXPERIMENT_NAME} for {config.num_epochs} epochs")
print("=" * 60)

for epoch in range(1, config.num_epochs + 1):
    epoch_start = time.time()
    torch.cuda.reset_peak_memory_stats()
    weight_snap = snapshot_weights(model)

    ep = train_one_epoch(model, train_loader, optimizer, noise_scheduler, scaler, ema,
                         timestep_sampler, config, global_step, total_steps)
    global_step = ep["global_step"]
    train_losses.append(ep["loss"])

    val_loss = validate(model, val_loader, noise_scheduler, config)
    val_losses.append(val_loss)

    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]["lr"]

    stats.record_epoch(
        train_loss=ep["loss"], val_loss=val_loss, lr=current_lr, epoch_time=epoch_time,
        gpu_allocated=torch.cuda.max_memory_allocated() / 1e9,
        gpu_reserved=torch.cuda.max_memory_reserved() / 1e9,
        gpu_nvidia_smi=get_gpu_memory_nvidia_smi(),
        data_time=ep["data_time"], forward_time=ep["fwd_time"], backward_time=ep["bwd_time"],
        grad_norm_avg=ep["grad_norm_avg"], grad_norm_max=ep["grad_norm_max"],
        grad_clip_count=ep["grad_clip_count"], nan_inf_count=ep["nan_inf_count"],
        ema_divergence=compute_ema_divergence(model, ema),
        loss_buckets=ep["loss_buckets"],
        weight_norm=compute_weight_norm(model),
        weight_update_ratio=compute_weight_update_ratio(model, weight_snap),
    )

    print(f"epoch {epoch:3d}/{config.num_epochs}  "
          f"train {ep['loss']:.4f}  val {val_loss:.4f}  "
          f"{epoch_time:.1f}s  lr {current_lr:.2e}")

    # visual preview every epoch
    preview = preview_with_ema()
    show_sample(preview, title=f"epoch {epoch}  loss={ep['loss']:.4f}",
                save_path=f"{config.output_path}/samples/epoch_{epoch:03d}.png")
    if epoch in SAMPLE_EPOCHS:
        save_nifti(preview, f"{config.output_path}/samples/epoch_{epoch:03d}.nii.gz")

    # checkpoints
    ckpt = {
        "epoch": epoch, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(), "ema": ema.state_dict(),
        "train_loss": ep["loss"], "val_loss": val_loss,
        "global_step": global_step, "config": config,
    }
    if epoch in SAMPLE_EPOCHS:
        path = f"{config.output_path}/checkpoints/epoch_{epoch}.pt"
        torch.save(ckpt, path)
        stats.stats["checkpoints"]["periodic_checkpoint_paths"].append(path)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        path = f"{config.output_path}/checkpoints/best_model.pt"
        torch.save({**ckpt, "best_epoch": epoch, "best_val_loss": best_val_loss}, path)
        stats.stats["checkpoints"]["best_checkpoint_path"] = path
        stats.stats["checkpoints"]["best_checkpoint_epoch"] = epoch
    if epoch == config.num_epochs:
        path = f"{config.output_path}/checkpoints/final_model.pt"
        torch.save(ckpt, path)
        stats.stats["checkpoints"]["final_checkpoint_path"] = path

    stats.save(f"{config.output_path}/stats.json")

stats.stats["training_info"]["end_time"] = datetime.now().isoformat()
stats.finalize()
stats.save(f"{config.output_path}/stats.json")

print(f"\ndone. stats saved to {config.output_path}/stats.json")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title(f"{EXPERIMENT_NAME} training")
plt.legend()
plt.grid(True)
plt.savefig(f"{config.output_path}/training_loss.png", dpi=150)
plt.show()

print(f"final  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")
print(f"best   val={min(val_losses):.4f} at epoch {val_losses.index(min(val_losses)) + 1}")


In [ ]:
print("generating final samples with EMA weights, each on a different val seg...")
ema.apply_shadow(model)
for i in range(4):
    _, seg = val_subset[i]
    seg = seg.unsqueeze(0).to(config.device)
    vol = sample(model, preview_scheduler, config.image_size,
                 num_steps=100, device=config.device, seed=i,
                 cond=seg, guidance_scale=config.guidance_scale)
    show_sample(vol, title=f"final sample {i + 1} (cond from val[{i}], guidance={config.guidance_scale})")
    save_nifti(vol, f"{config.output_path}/samples/final_sample_{i}.nii.gz")
    save_nifti(seg.squeeze(0), f"{config.output_path}/samples/final_cond_{i}.nii.gz")
ema.restore(model)
